<a href="https://colab.research.google.com/github/Mufliha-Hafsyah/Machine-Learning/blob/main/Week06/Praktikum4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM PEMBELAJARAN MESIN**

**Nama**  : Mufliha Hafsyah Shahieza <br>
**NIM**   : 244107020147 <br>
**Kelas** : TI-3G

---

# **JS06 - ANN (Approximate Nearest Neighbors)**<hr>
# **Praktikum 4**<hr>

In [ ]:
!pip install annoy faiss-cpu hnswlib

  Using cached annoy-1.17.3-cp313-cp313-linux_x86_64.whl
  Using cached faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
  Using cached hnswlib-0.8.0.tar.gz (36 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.8 MB)
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679068 sha256=b30904554504266c9630705b69ae5037bb3877365725fd2fa5192eb18cbb151c
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [ ]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

def percobaan(metric, n_data=1_000_000, dim=5, k=10, seed=42):
    np.random.seed(seed)
    X = np.random.random((n_data, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)

    annoy_metric = 'euclidean' if metric == 'l2' else 'angular'
    hnsw_space = 'l2' if metric == 'l2' else 'cosine'

    # FAISS Flat (exact) = acuan
    if metric == 'l2':
        Xf, qf = X, query
        fa = faiss.IndexFlatL2(dim)
    else:
        Xf, qf = X.copy(), query.copy()
        faiss.normalize_L2(Xf)
        faiss.normalize_L2(qf)
        fa = faiss.IndexFlatIP(dim)
    t = time.time(); fa.add(Xf); b_fa = time.time() - t
    t = time.time(); _, I_fa = fa.search(qf, k); q_fa = time.time() - t
    exact = set(I_fa[0].tolist())

    # Annoy
    an = AnnoyIndex(dim, annoy_metric)
    t = time.time()
    for i in range(n_data):
        an.add_item(i, X[i])
    an.build(10)
    b_an = time.time() - t
    t = time.time(); I_an = an.get_nns_by_vector(query[0], k); q_an = time.time() - t

    # HNSW
    hn = hnswlib.Index(space=hnsw_space, dim=dim)
    t = time.time()
    hn.init_index(max_elements=n_data, ef_construction=200, M=16)
    hn.add_items(X)
    b_hn = time.time() - t
    hn.set_ef(50)
    t = time.time(); I_hn, _ = hn.knn_query(query, k=k); q_hn = time.time() - t

    print(f"[{metric} | {n_data:,} data | {dim}D | k={k}]")
    for nama, idx, b, q in [("Annoy (10 tree)", I_an, b_an, q_an),
                            ("FAISS Flat", I_fa[0].tolist(), b_fa, q_fa),
                            ("HNSW", I_hn[0].tolist(), b_hn, q_hn)]:
        rec = len(exact & set(idx)) / k
        print(f"  {nama:16s} build={b:8.3f} s | query={q*1000:8.4f} ms | recall@{k}={rec:.1f} | top5={list(idx)[:5]}")
    print()

for m in ('l2', 'cosine'):
    percobaan(m)

[l2 | 1,000,000 data | 5D | k=10]
  Annoy (10 tree)  build=  12.795 s | query=  0.1628 ms | recall@10=0.9 | top5=[901095, 495168, 561017, 163986, 185734]
  FAISS Flat       build=   0.010 s | query=  8.5549 ms | recall@10=1.0 | top5=[901095, 495168, 561017, 163986, 185734]
  HNSW             build= 155.393 s | query=  0.3064 ms | recall@10=1.0 | top5=[901095, 495168, 561017, 163986, 185734]

[cosine | 1,000,000 data | 5D | k=10]
  Annoy (10 tree)  build=  16.192 s | query=  0.1140 ms | recall@10=1.0 | top5=[653774, 135588, 561017, 901095, 211473]
  FAISS Flat       build=   0.012 s | query=  3.2952 ms | recall@10=1.0 | top5=[653774, 135588, 561017, 901095, 211473]
  HNSW             build= 137.989 s | query=  0.2210 ms | recall@10=1.0 | top5=[653774, 135588, 561017, 901095, 211473]



### **Hasil Percobaan 4: Annoy vs FAISS vs HNSW (1.000.000 data, 5D, k=10)**

| Distance Metrics | Metode | Waktu build (detik) | Waktu query (ms) | Recall@10 | 5 Index terdekat |
|---|---|---|---|---|---|
| L2 (Euclidean) | Annoy (10 tree) | 12.795 | 0.1628 | 0.9 | [901095, 495168, 561017, 163986, 185734] |
| L2 (Euclidean) | FAISS Flat | 0.010 | 8.5549 | 1.0 (acuan) | [901095, 495168, 561017, 163986, 185734] |
| L2 (Euclidean) | HNSW | 155.393 | 0.3064 | 1.0 | [901095, 495168, 561017, 163986, 185734] |
| Cosine / Angular | Annoy (10 tree) | 16.192 | 0.1140 | 1.0 | [653774, 135588, 561017, 901095, 211473] |
| Cosine / Angular | FAISS Flat | 0.012 | 3.2952 | 1.0 (acuan) | [653774, 135588, 561017, 901095, 211473] |
| Cosine / Angular | HNSW | 137.989 | 0.2210 | 1.0 | [653774, 135588, 561017, 901095, 211473] |

**Analisis:**

1. **Kecepatan query.** Annoy dan HNSW jauh lebih cepat dari FAISS Flat (brute force). Pada L2, Annoy sekitar 52 kali dan HNSW sekitar 28 kali lebih cepat; pada Cosine, Annoy sekitar 29 kali dan HNSW sekitar 15 kali lebih cepat. Pada percobaan ini Annoy sedikit lebih cepat daripada HNSW saat query.
2. **Waktu build.** Kecepatan query dibayar di waktu pembangunan indeks. FAISS Flat hampir tanpa build (hanya menyalin data, sekitar 0,01 detik), Annoy sekitar 13 sampai 16 detik, dan HNSW paling lama, sekitar 138 sampai 155 detik (kira-kira 9 sampai 12 kali lebih lama dari Annoy).
3. **Akurasi.** FAISS Flat adalah pencarian eksak sehingga menjadi acuan (recall 1.0). HNSW mencapai recall 1.0 pada kedua metrik. Annoy mencapai recall 0.9 pada L2 (9 dari 10 tetangga sama, 5 teratas identik) dan 1.0 pada Cosine. Annoy bersifat acak (pohon dibangun acak), sehingga recall-nya bisa sedikit berbeda tiap dijalankan; akurasinya dapat ditingkatkan dengan menambah jumlah tree.
4. **Pengaruh metrik jarak.** Pada data yang sama, L2 dan Cosine menghasilkan tetangga terdekat yang berbeda (hanya index 901095 dan 561017 yang muncul di kedua daftar 5 teratas), karena L2 mengukur jarak absolut sedangkan Cosine hanya mengukur arah vektor.
5. **Kesimpulan.** FAISS Flat cocok untuk data kecil atau saat butuh hasil eksak tanpa persiapan. Annoy memberi keseimbangan yang baik antara waktu build yang singkat dan query yang cepat. HNSW memberi akurasi terbaik dengan query yang sangat cepat, tetapi dengan waktu build terpanjang dan memori yang lebih besar.